In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md



/kaggle/input/competitions/house-prices-advanced-regression-techniques/sample_submission.csv
/kaggle/input/competitions/house-prices-advanced-regression-techniques/data_description.txt
/kaggle/input/competitions/house-prices-advanced-regression-techniques/train.csv
/kaggle/input/competitions/house-prices-advanced-regression-techniques/test.csv


In [2]:
# ============================================================
# KAGGLE HOUSE PRICES - COMPLETE SUBMISSION NOTEBOOK
# ============================================================

import os
import glob
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import mean_squared_error
from sklearn.ensemble import GradientBoostingRegressor


# ============================================================
# 1. FIND train.csv AND test.csv AUTOMATICALLY
# ============================================================

print("Searching for dataset files...\n")

train_files = glob.glob("/kaggle/input/**/train.csv", recursive=True)
test_files = glob.glob("/kaggle/input/**/test.csv", recursive=True)

print("Train files found:")
for f in train_files:
    print(f)

print("\nTest files found:")
for f in test_files:
    print(f)


if len(train_files) == 0:
    raise FileNotFoundError(
        "train.csv was not found in /kaggle/input/. "
        "Please add the House Prices competition dataset."
    )

if len(test_files) == 0:
    raise FileNotFoundError(
        "test.csv was not found in /kaggle/input/. "
        "Please add the House Prices competition dataset."
    )


# Use the first matching files
TRAIN_PATH = train_files[0]
TEST_PATH = test_files[0]

print("\nUsing:")
print("TRAIN:", TRAIN_PATH)
print("TEST :", TEST_PATH)


# ============================================================
# 2. LOAD DATA
# ============================================================

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)

print("\nTrain shape:", train.shape)
print("Test shape :", test.shape)

print("\nTrain columns:", len(train.columns))
print("Test columns :", len(test.columns))


# ============================================================
# 3. CHECK REQUIRED COLUMNS
# ============================================================

if "SalePrice" not in train.columns:
    raise ValueError("SalePrice column is missing from train.csv")

if "Id" not in train.columns:
    raise ValueError("Id column is missing from train.csv")

if "Id" not in test.columns:
    raise ValueError("Id column is missing from test.csv")


# ============================================================
# 4. SAVE TEST IDS
# ============================================================

test_ids = test["Id"].copy()


# ============================================================
# 5. SEPARATE FEATURES AND TARGET
# ============================================================

X = train.drop(columns=["SalePrice", "Id"])
X_test = test.drop(columns=["Id"])

# Kaggle evaluates logarithmic RMSE.
# Therefore train using log1p(SalePrice).

y = np.log1p(train["SalePrice"])


# ============================================================
# 6. IDENTIFY NUMERIC / CATEGORICAL COLUMNS
# ============================================================

numeric_features = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X.select_dtypes(
    include=["object"]
).columns.tolist()

print("\nNumeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))


# ============================================================
# 7. PREPROCESSING
# ============================================================

numeric_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        )
    ]
)


categorical_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            )
        )
    ]
)


preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_transformer,
            numeric_features
        ),
        (
            "categorical",
            categorical_transformer,
            categorical_features
        )
    ]
)


# ============================================================
# 8. MODEL
# ============================================================

model = GradientBoostingRegressor(
    n_estimators=1000,
    learning_rate=0.03,
    max_depth=4,
    max_features="sqrt",
    min_samples_leaf=10,
    min_samples_split=10,
    loss="huber",
    random_state=42
)


# ============================================================
# 9. CREATE PIPELINE
# ============================================================

pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", model)
    ]
)


# ============================================================
# 10. VALIDATION
# ============================================================

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)


print("\nTraining validation model...")

pipeline.fit(
    X_train,
    y_train
)


# ============================================================
# 11. VALIDATION SCORE
# ============================================================

valid_predictions = pipeline.predict(X_valid)

validation_rmse = np.sqrt(
    mean_squared_error(
        y_valid,
        valid_predictions
    )
)

print(
    f"\nValidation Log RMSE: "
    f"{validation_rmse:.5f}"
)


# ============================================================
# 12. TRAIN FINAL MODEL ON ALL TRAINING DATA
# ============================================================

print("\nTraining final model on complete training dataset...")

pipeline.fit(
    X,
    y
)

print("Final model training completed.")


# ============================================================
# 13. PREDICT TEST SET
# ============================================================

print("\nGenerating predictions...")

test_log_predictions = pipeline.predict(
    X_test
)


# Convert log predictions back to SalePrice
test_predictions = np.expm1(
    test_log_predictions
)


# Make sure no negative prices exist
test_predictions = np.maximum(
    test_predictions,
    0
)


# ============================================================
# 14. CREATE SUBMISSION DATAFRAME
# ============================================================

submission = pd.DataFrame(
    {
        "Id": test_ids,
        "SalePrice": test_predictions
    }
)


# ============================================================
# 15. VALIDATE SUBMISSION
# ============================================================

print("\nSubmission preview:")
display(submission.head(10))

print("\nSubmission shape:")
print(submission.shape)

print("\nExpected test rows:")
print(len(test))

print("\nSubmission columns:")
print(submission.columns.tolist())


# Check number of rows
assert len(submission) == len(test)

# Check required columns
assert list(submission.columns) == [
    "Id",
    "SalePrice"
]

# Check for missing values
assert submission["Id"].isna().sum() == 0
assert submission["SalePrice"].isna().sum() == 0

# Check SalePrice values
assert (submission["SalePrice"] >= 0).all()


# ============================================================
# 16. SAVE SUBMISSION FILE
# ============================================================

OUTPUT_PATH = "/kaggle/working/submission.csv"

submission.to_csv(
    OUTPUT_PATH,
    index=False
)


# ============================================================
# 17. VERIFY FILE EXISTS
# ============================================================

print("\n" + "=" * 60)
print("SUBMISSION FILE CREATED")
print("=" * 60)

print("\nFile path:")
print(OUTPUT_PATH)

print("\nFile exists:")
print(os.path.exists(OUTPUT_PATH))

print("\nFile size:")
print(os.path.getsize(OUTPUT_PATH), "bytes")


# ============================================================
# 18. READ FILE BACK AND VERIFY
# ============================================================

check_submission = pd.read_csv(
    OUTPUT_PATH
)

print("\nSubmission file:")
display(check_submission.head())

print("\nSubmission rows:", len(check_submission))
print("Submission columns:", check_submission.columns.tolist())


# ============================================================
# 19. DISPLAY FINAL FILE CONTENT
# ============================================================

print("\nFinal submission:")
display(check_submission.head(20))






Searching for dataset files...

Train files found:
/kaggle/input/competitions/house-prices-advanced-regression-techniques/train.csv

Test files found:
/kaggle/input/competitions/house-prices-advanced-regression-techniques/test.csv

Using:
TRAIN: /kaggle/input/competitions/house-prices-advanced-regression-techniques/train.csv
TEST : /kaggle/input/competitions/house-prices-advanced-regression-techniques/test.csv

Train shape: (1460, 81)
Test shape : (1459, 80)

Train columns: 81
Test columns : 80

Numeric features: 36
Categorical features: 43

Training validation model...

Validation Log RMSE: 0.13283

Training final model on complete training dataset...
Final model training completed.

Generating predictions...

Submission preview:


,Id,SalePrice
0,1461,124510.868299
1,1462,156530.093300
2,1463,189883.669181
3,1464,192784.733501
4,1465,186531.681129
5,1466,174417.069247
6,1467,174498.213728
7,1468,167244.330322
8,1469,185158.509304
9,1470,128395.050490



Submission shape:
(1459, 2)

Expected test rows:
1459

Submission columns:
['Id', 'SalePrice']

SUBMISSION FILE CREATED

File path:
/kaggle/working/submission.csv

File exists:
True

File size:
34460 bytes

Submission file:


,Id,SalePrice
0,1461,124510.868299
1,1462,156530.093300
2,1463,189883.669181
3,1464,192784.733501
4,1465,186531.681129



Submission rows: 1459
Submission columns: ['Id', 'SalePrice']

Final submission:


,Id,SalePrice
0,1461,124510.868299
1,1462,156530.093300
2,1463,189883.669181
3,1464,192784.733501
4,1465,186531.681129
5,1466,174417.069247
6,1467,174498.213728
7,1468,167244.330322
8,1469,185158.509304
9,1470,128395.050490
